In [ ]:
!pip install -U transformers datasets accelerate peft bitsandbytes trl
!pip install protobuf==3.20.3 "pyarrow<20.0.0"

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftConfig, get_peft_model, PeftModel
from trl import SFTTrainer
from trl import setup_chat_format
from transformers import (AutoModelForCausalLM, 
                          AutoTokenizer, 
                          BitsAndBytesConfig, 
                          TrainingArguments, 
                          pipeline, 
                          logging)
from sklearn.metrics import (accuracy_score, 
                             classification_report, 
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [ ]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN") 
os.environ["HF_TOKEN"] = hf_token

login(token=hf_token)

In [ ]:
raw_datasets = load_dataset("ailsntua/QEvasion")

train_df_full = raw_datasets['train'].to_pandas()

train_df, val_df = train_test_split(
    train_df_full,
    test_size=700,
    random_state=42,
    stratify=train_df_full['clarity_label']
)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = raw_datasets['test']

label2id = {
    "Clear Reply": 0,
    "Ambivalent": 1,
    "Clear Non-Reply": 2
}

id2label = {v: k for k, v in label2id.items()}

In [ ]:
base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16, 
    quantization_config=bnb_config, 
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" 

In [ ]:
def format_chat_template(example):
    question = example['question']
    answer = example['interview_answer']
    clarity_label = example['clarity_label']
    conversation = [
        {
            "role": "system",
            "content": "You are an expert in analyzing political discourse. Classify the clarity of responses into: Clear Reply, Ambivalent, or Clear Non-Reply. Output ONLY the label name. Do not explain."
        },
        {
            "role": "user",
            "content": f"Question: {question}\nAnswer: {answer}\n\nClassify the clarity of this response."
        },
        {
            "role": "assistant",
            "content": clarity_label
        }
    ]
    
    return {"conversation": conversation}

def apply_chat_template_to_str(example):
    formatted_text = tokenizer.apply_chat_template(
        example["conversation"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"formatted_chat": formatted_text}

train_dataset = train_dataset.map(format_chat_template)
val_dataset = val_dataset.map(format_chat_template)

train_dataset = train_dataset.map(apply_chat_template_to_str)
val_dataset = val_dataset.map(apply_chat_template_to_str)

test_dataset = test_dataset.map(format_chat_template)
test_dataset = test_dataset.map(apply_chat_template_to_str)

In [ ]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    
    return list(lora_module_names)

target_modules = find_all_linear_names(model)
print(f"Auto-detected LoRA target modules: {target_modules}")

In [ ]:
def evaluate_model(trainer, dataset, dataset_name="validation", batch_size=8):
    """
    Evaluate model using batched generation for speed
    batch_size=8 is optimal for T4/P100 GPUs with 4-bit models
    """
    print(f"\nEvaluating on {dataset_name} set (batch_size={batch_size})...")
    
    predictions = []
    true_labels = []
    
    prompts = []
    for example in dataset:
        conversation = [
            {
                "role": "system",
                "content": "You are an expert in analyzing political discourse. Classify the clarity of responses into: Clear Reply, Ambivalent, or Clear Non-Reply. Output ONLY the label name. Do not explain."
            },
            example['conversation'][1]  # user message
        ]
        
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(text)
        true_labels.append(example['clarity_label'])
    
    # Step 2: Process in batches (8 examples at once)
    for i in tqdm(range(0, len(prompts), batch_size), desc="Batch Generation"):
        batch_prompts = prompts[i : i + batch_size]
        
        # Tokenize the whole batch at once
        # padding=True ensures all sequences are same length (critical for batching)
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)
        
        with torch.no_grad():
            with torch.autocast("cuda"): 
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=20,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )
        
        generated_texts = tokenizer.batch_decode(
            outputs[:, inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        
        for text in generated_texts:
            clean_text = text.strip()
            
            if "Clear Reply" in clean_text or "clear reply" in clean_text.lower():
                pred = "Clear Reply"
            elif "Ambivalent" in clean_text or "ambivalent" in clean_text.lower():
                pred = "Ambivalent"
            elif "Clear Non-Reply" in clean_text or "clear non-reply" in clean_text.lower():
                pred = "Clear Non-Reply"
            else:
                pred = "Ambivalent"  # Fallback to most common class
            
            predictions.append(pred)
    
    # Step 3: Calculate metrics
    accuracy = accuracy_score(true_labels, predictions)
    macro_f1 = f1_score(
        true_labels,
        predictions,
        average='macro',
        labels=["Clear Reply", "Ambivalent", "Clear Non-Reply"],
        zero_division=0
    )
    
    print(f"\n{dataset_name.capitalize()} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}\n")
    print(classification_report(
        true_labels,
        predictions,
        labels=["Clear Reply", "Ambivalent", "Clear Non-Reply"],
        zero_division=0
    ))
    
    return predictions, true_labels, accuracy, macro_f1

In [ ]:
peft_config = LoraConfig(
    r=16,                      
    lora_alpha=32,             
    lora_dropout=0.05,         
    bias="none",               
    task_type="CAUSAL_LM",     
    target_modules=target_modules  
)

from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./llama3.1_improved_results",
    
    max_length=2048,
    dataset_text_field="formatted_chat",
    packing=True,
    
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    
    fp16=True,
    max_grad_norm=0.3,
    
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,

    push_to_hub=True,                        
    hub_model_id="gabrielstefan04/llama-3-1-8b-clarity", 
    hub_strategy="every_save",               
    hub_token=os.environ["HF_TOKEN"],
)

trainer = SFTTrainer(
    model=model,
    args=training_args,              
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,      
)

In [ ]:
trainer.train()

In [ ]:
tokenizer.padding_side = 'left' 
val_predictions, val_true_labels, val_accuracy, val_f1 = evaluate_model(trainer, val_dataset, "validation")

In [ ]:
test_predictions, test_true_labels, test_accuracy, test_f1 = evaluate_model(trainer, test_dataset, "test")